In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: Script en Python para generar heatmaps noches frías por municipio para la ZMVM. 
# Periodo: 2000-2024
# ==========================================

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Cargar datos
ruta_csv = "RUTA ARCHIVO"
df = pd.read_csv(ruta_csv)

# Normalizar CVEGEO a mayúsculas
df["CVEGEO"] = df["CVEGEO"].str.upper()

# Lista de municipios disponibles
delegaciones = df["CVEGEO"].unique()
delegaciones.sort()
print("Delegaciones disponibles:")
for d in delegaciones:
    print("-", d)


indice = "TN10p"   

# Carpeta de salida
carpeta_salida = "RUTA DE SALIDA"
os.makedirs(carpeta_salida, exist_ok=True)

# Lista de meses para eje Y
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

# Generar un heatmap por cada municipio
for municipio in delegaciones:
    df_muni = df[df["CVEGEO"] == municipio].copy()

    if df_muni.empty:
        continue  # Si no hay datos, saltar

    # Pivotear: filas = mes, columnas = año
    pivot_idx = df_muni.pivot_table(index="mes", columns="anio", values=indice, fill_value=0)
    pivot_idx = pivot_idx.sort_index()  # ordena meses

    # Anotaciones con %
    annot_percent = pivot_idx.map(lambda x: f"{int(round(x))}%" if pd.notnull(x) else "")

    # Crear heatmap
    plt.figure(figsize=(14, 8))
    ax = sns.heatmap(
        pivot_idx,
        cmap="YlOrRd" if "90" in indice else "Blues",  
        linewidths=0.2,
        linecolor="white",
        annot=annot_percent,
        fmt="",
        annot_kws={"size": 7},
        cbar_kws={"format": "%.0f%%"}
    )

    # Formatear barra de color
    colorbar = ax.collections[0].colorbar
    colorbar.set_ticks(colorbar.get_ticks())
    colorbar.set_ticklabels([f"{int(t)}%" for t in colorbar.get_ticks()])

    # Etiquetas
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)  # Años en X
    ax.set_yticks(np.arange(12) + 0.5)
    ax.set_yticklabels(meses, rotation=0)

    plt.xlabel("Año")
    plt.ylabel("Mes")

    # Título
    plt.title(f"ÍNDICE MENSUAL ({indice})\n{municipio}", fontsize=16, fontweight="bold")

    # Guardar figura
    nombre_archivo = f"{indice}_{municipio}.png"
    plt.tight_layout()
    plt.savefig(os.path.join(carpeta_salida, nombre_archivo), dpi=300, bbox_inches="tight")
    plt.close()  # cerrar para no sobrecargar memoria

print(f"Se generaron {len(delegaciones)} gráficos en la carpeta: {carpeta_salida}")


Delegaciones disponibles:
- ACOLMAN
- ALVARO OBREGON
- ATIZAPAN DE ZARAGOZA
- AZCAPOTZALCO
- BENITO JUAREZ
- CHALCO
- COACALCO DE BERRIOZABAL
- COYOACAN
- CUAJAMILPA DE MORELOS
- CUAUHTEMOC
- CUAUTITLAN DE IZCALLI
- ECATEPEC
- GUSTAVO A. MADERO
- IZTALCO
- IZTAPALAPA
- MIGUEL HIDALGO
- MILPA ALTA
- NAUCALPAN DE JUAREZ
- NEZAHUALCOYOTL
- TLALPAN
- TLANEPANTLA
- TULTITLAN
- VENUSTIANO CARRANZA
Se generaron 23 gráficos en la carpeta: /Users/arelyleal/Downloads/TESIS/BASES DE DATOS/CAMBIO CLIMATICO/PRUEBA
